**Cell 1**

# Role 3 - Preprocessing and signal validation

**Owner: unassigned**

Own the complete preprocessing chain: PSD, whitening, bandpass, crop, and numerical/physical checks. This is the dedicated preprocessing role. You do not acquire production data, create labels or train models. Your checkpoint deliverable is data evidence, not a trained model.

Revised scope: dataset preparation, preprocessing and EDA. This is a new notebook; earlier role notebooks are preserved. The implementation cells now provide a reproducible synthetic validation path and a conditional real-data handoff path.

**Cell 2**

## What this milestone covers

The project question remains whether four-parameter inference generalizes from O3a H1 noise to O4a H1 noise. This checkpoint concentrates on preparing and understanding the dataset: source provenance, physical labels, preprocessing, actual split counts, EDA, plots, and limitations. Model training, model performance tables and calibration conclusions are outside this notebook set's required work.

The methodology section must still describe three algorithms because the rubric asks for a baseline and two improvements: linear regression, its small regularized extension Ridge, and SNPE with a masked autoregressive flow. Describe them as the next common comparison; do not invent new training results. Feature importance is explicitly requested by the rubric. Input-target association is useful EDA but does not fully substitute for model-based importance; Role 4 must state this gap honestly if importance is deferred.

**Existing evidence versus your work:** The repository already contains saved noise, injections and processed data. The revised PDF uses those artifacts, not results from these new notebooks. Your task is a separately identified manual pilot. Do not claim your pilot has the old dataset's 800-per-run counts until you actually generate and check them. The earlier split was a 750-row O3a development pool, 50 O3a held-out examples and 800 O4a evaluation examples. A 600/150 subdivision of the development pool is a proposal, not a completed split.

**Implementation rule:** Write your own code in the empty cells. You may call standard scientific libraries and consult their documentation, but do not import ligo_pipeline, project utils, config.py, another notebook's implementation, or prepared training/model checkpoints. Trusted raw strain with source metadata is allowed. You may write your own small functions after explaining their steps. No solution code is supplied here.

**Ownership:** Assign one person per notebook; names are unassigned. Review each other's outputs, but let each owner make their own notebook changes. Record actual contributions and dates. Every cell has a sequential cell_number and every markdown cell displays its number; the preceding task labels the empty code cell. Renumber all cells from 1 when inserting new cells. Execution counters are not stable cell numbering.


**Cell 3**

## Common contract and parallel work

Agree on one small experiment before scaling up: H1; O3a development and within-run evaluation; O4a cross-run evaluation; 4096 Hz; four-second final windows; a 35-350 Hz preprocessing band; target order chirp_mass, mass_ratio, chi_eff, distance. Keep any fixed-mass, distance-only exercise separate from this four-target pilot.

Use a new experiment_id and store files under artifacts/manual_preprocessing_milestone/<experiment_id>/. The role-owned subfolders are data/, simulation/, preprocessing/ and analysis/. Put experiment.json in data/ and repeat its ID and relevant settings in each output's metadata. Store raw segment duration, sample rate, waveform assumptions, parameter ranges, seeds, intended counts, actual counts and units. Do not overwrite the existing artifacts or the older role notebooks.

Role 1 supplies raw arrays, noise_id/group_id, source GPS intervals and source split assignments. Role 3 supplies noise-only PSD files early. Role 2 uses those PSDs for SNR selection, then supplies padded injected buffers, merger indices and labels. Role 3 supplies processed arrays and row metadata. Role 4 audits and combines evidence, counts and EDA; this role does not train the final models.

**Source-group separation:** Assign groups before simulation. Overlapping raw GPS intervals are one group even if filenames differ. Use train, validation, O3a_test and O4a_test split names; keep all O4a out of fitting and tuning. A recommended small design has at least three O3a training groups, one O3a validation group, one O3a test group and two O4a groups. Choose feasible durations and target counts together; these are planning suggestions, not completed allocations. If your pilot cannot support this design, document the limitation. Track nonoverlapping final crop windows as well as group identities.

**Handoff format:** Use .npy/.npz numeric arrays and .csv/.json metadata with repo-relative paths. Join per-example records by unique sample_id; join noise/PSD records by noise_id. Preserve run, split, group and units. Never assume two files have matching row order. Save rejection ledgers, shapes, dtypes, versions and seeds. Changes to schema/settings require all consumers to check compatibility.

**Start in parallel:** Role 1 can inspect one raw segment, Role 2 can inspect clean waveforms, Role 3 can validate sinusoids/FFT/filtering, and Role 4 can design the counting ledger, plot checklist and split audit. First integrate one real example end to end through data preparation. Then expand the pilot; no model fit is needed to check these handoffs.


**Cell 4**

## Your work boundary

**Inputs:** Role 1 raw noise and source metadata; Role 2 padded injected buffers with merger indices and target IDs. Until those arrive, use simple sinusoids/noise and one real raw segment to learn the transformations.

**Outputs:** preprocessing/psd/<noise_id>.npz; preprocessing/processed_manifest.csv; preprocessing/processed.npz; preprocessing/rejections.csv; before/after plots and processing_settings.json. Deliver PSD files early so Role 2 can calculate SNR.

**Start in parallel:** You can study FFTs, validate frequency axes, design a filter and implement crop boundary checks immediately. Export a noise PSD as soon as Role 1 supplies one segment. Finish real whitening/cropping once Role 2 supplies one padded injection. The full dataset is not needed to prove your methods.

**File contract:** Each PSD .npz contains frequency_hz and psd_strain2_per_hz and is identified by noise_id; its metadata records source and Welch settings. processed.npz contains X with shape (N,16384), theta with shape (N,4), sample_id and split arrays. processed_manifest.csv adds row_index, sample_id, noise_id, group_id, run, split, n_samples, finite_status and source buffer path. processed arrays and manifests must agree exactly; use explicit numeric/string dtypes, not pickled objects.

All paths use the experiment-specific root in the common contract above. Older path wording in conceptual examples does not authorize writing into historical artifact folders.

**Cell 5**

## Task 1: Build the time and frequency foundations

### Background knowledge
FFT bins are fk = k*fs/N, the Nyquist frequency is fs/2, and frequency resolution is fs/N. A raw FFT magnitude is not a PSD. For real inputs, a one-sided transform avoids duplicate negative-frequency bins. Windowing changes spectral leakage.

### Your task
Construct a known sinusoid with a duration and sample rate you choose. Compute its frequency axis and show where its peak should be. Compare windowed/unwindowed spectra, then inspect one real segment. Explain bin spacing, Nyquist and the 35-350 Hz analysis band.

### Evidence and completion checks
Time/spectrum plots and a measured sinusoid peak error; axes include seconds, Hz and units.

**Cell 6 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 6
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ligo-parameter-inference repository.")

PROJECT_ROOT = find_project_root(Path.cwd())

# Team-owned contract: change EXPERIMENT_ID only after all four role owners agree.
EXPERIMENT_ID = "manual_preprocessing_v1"
EXPERIMENT_ROOT = PROJECT_ROOT / "artifacts" / "manual_preprocessing_milestone" / EXPERIMENT_ID
DATA_DIR = EXPERIMENT_ROOT / "data"
SIMULATION_DIR = EXPERIMENT_ROOT / "simulation"
PREPROCESSING_DIR = EXPERIMENT_ROOT / "preprocessing"

SAMPLE_RATE_HZ = 4096
FINAL_WINDOW_SEC = 4.0
HALF_WINDOW_SEC = FINAL_WINDOW_SEC / 2
WINDOW_SAMPLES = int(FINAL_WINDOW_SEC * SAMPLE_RATE_HZ)
LOW_HZ = 35.0
HIGH_HZ = 350.0
FILTER_ORDER = 4
PSD_NPERSEG_SEC = 4.0
PSD_OVERLAP_FRACTION = 0.5
PSD_FLOOR_RELATIVE = 1e-12
REQUIRED_PADDING_SEC = 15.0
TARGET_NAMES = ["chirp_mass", "mass_ratio", "chi_eff", "distance"]
SEED = 207

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def resolve_repo_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

# Known-frequency validation: the nearest FFT bin must recover the input tone.
tone_duration_sec = 2.0
tone_frequency_hz = 123.25
tone_time_sec = np.arange(int(tone_duration_sec * SAMPLE_RATE_HZ)) / SAMPLE_RATE_HZ
tone = np.sin(2 * np.pi * tone_frequency_hz * tone_time_sec)
tone_frequencies_hz = np.fft.rfftfreq(tone.size, d=1 / SAMPLE_RATE_HZ)
raw_spectrum = np.abs(np.fft.rfft(tone))
hann_window = signal.windows.hann(tone.size, sym=False)
windowed_spectrum = np.abs(np.fft.rfft(tone * hann_window))
measured_peak_hz = tone_frequencies_hz[np.argmax(raw_spectrum)]
frequency_resolution_hz = SAMPLE_RATE_HZ / tone.size
peak_error_hz = abs(measured_peak_hz - tone_frequency_hz)
assert peak_error_hz <= frequency_resolution_hz / 2 + 1e-12

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
axes[0].plot(tone_time_sec[: int(0.05 * SAMPLE_RATE_HZ)], tone[: int(0.05 * SAMPLE_RATE_HZ)])
axes[0].set(title="Known sinusoid in time", xlabel="Time (s)", ylabel="Amplitude (arbitrary units)")
zoom = (tone_frequencies_hz >= 110) & (tone_frequencies_hz <= 136)
axes[1].plot(tone_frequencies_hz[zoom], raw_spectrum[zoom] / raw_spectrum.max(), label="Rectangular window")
axes[1].plot(tone_frequencies_hz[zoom], windowed_spectrum[zoom] / windowed_spectrum.max(), label="Hann window")
axes[1].axvline(tone_frequency_hz, color="black", linestyle="--", linewidth=1, label="True frequency")
axes[1].set(title="Spectral leakage check", xlabel="Frequency (Hz)", ylabel="Normalized magnitude")
axes[1].legend()
plt.show()

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"FFT bin spacing: {frequency_resolution_hz:.3f} Hz; Nyquist: {SAMPLE_RATE_HZ / 2:.1f} Hz")
print(f"Known tone: {tone_frequency_hz:.3f} Hz; measured peak: {measured_peak_hz:.3f} Hz; error: {peak_error_hz:.3f} Hz")
print(f"Analysis band: {LOW_HZ:.1f}-{HIGH_HZ:.1f} Hz")

NOISE_MANIFEST_PATH = DATA_DIR / "noise_manifest.csv"
INJECTION_MANIFEST_PATH = SIMULATION_DIR / "injection_manifest.csv"
TARGETS_PATH = SIMULATION_DIR / "targets.csv"
REAL_HANDOFFS_AVAILABLE = all(path.is_file() for path in [NOISE_MANIFEST_PATH, INJECTION_MANIFEST_PATH, TARGETS_PATH])
print("Real Role 1/2 handoffs available:", REAL_HANDOFFS_AVAILABLE)
if not REAL_HANDOFFS_AVAILABLE:
    print("Synthetic validation will run. Real pilot export remains pending until the three manifest files exist.")


**Cell 7**

### Your findings and contribution record

**Implementation status:** A deterministic known-frequency test now verifies the time axis, FFT bin spacing, Nyquist frequency, windowing behavior and measured peak error. The cell also fixes the common experiment contract and detects whether the Role 1/2 manifests are present. It does not claim a real-noise finding when those handoffs are absent.

**Handoff:** Run the cell and retain the printed frequency error and labeled time/frequency figure. Confirm `EXPERIMENT_ID` with the team before producing shared artifacts.


**Cell 8**

## Task 2: Estimate a noise-only PSD and export it early

### Background knowledge
Welch PSD averages overlapping windowed periodograms and has units strain squared per Hz. ASD is its square root. More averaging reduces variability at the expense of frequency resolution. Estimate background PSD from the source noise, not from a buffer containing the injected signal.

### Your task
Implement and inspect a small Welch-style calculation, then compare with a trusted library result. Use four-second subwindows when the raw segment is long enough; record overlap, window and detrending. Check increasing frequency, nonnegative PSD and units. Explain a positive floor for unusable/near-zero bins. Export the first PSD and its settings to Role 2.

### Evidence and completion checks
A trustworthy PSD artifact, PSD/ASD plot, normalization check and documented floor/invalid-bin policy.

**Cell 9 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 9
def validate_1d_finite(values, name, minimum_length=1):
    array = np.asarray(values, dtype=np.float64)
    if array.ndim != 1:
        raise ValueError(f"{name} must be one-dimensional; got shape {array.shape}.")
    if array.size < minimum_length:
        raise ValueError(f"{name} has {array.size} samples; expected at least {minimum_length}.")
    if not np.isfinite(array).all():
        raise ValueError(f"{name} contains NaN or infinite values.")
    return array

def manual_welch_psd(values, sample_rate_hz, nperseg, noverlap):
    """Small explicit Welch implementation for normalization verification."""
    values = validate_1d_finite(values, "manual_welch_input", nperseg)
    step = nperseg - noverlap
    starts = np.arange(0, values.size - nperseg + 1, step)
    if starts.size == 0:
        raise ValueError("No complete Welch segments are available.")
    window = signal.windows.hann(nperseg, sym=False)
    window_power = np.sum(window**2)
    periodograms = []
    for start in starts:
        segment_values = values[start:start + nperseg]
        segment_values = signal.detrend(segment_values, type="constant")
        spectrum = np.fft.rfft(segment_values * window)
        density = np.abs(spectrum) ** 2 / (sample_rate_hz * window_power)
        if nperseg % 2 == 0:
            density[1:-1] *= 2
        else:
            density[1:] *= 2
        periodograms.append(density)
    frequencies_hz = np.fft.rfftfreq(nperseg, d=1 / sample_rate_hz)
    return frequencies_hz, np.mean(periodograms, axis=0)

def estimate_psd_welch(values, sample_rate_hz=SAMPLE_RATE_HZ,
                       nperseg_sec=PSD_NPERSEG_SEC,
                       overlap_fraction=PSD_OVERLAP_FRACTION,
                       floor_relative=PSD_FLOOR_RELATIVE):
    values = validate_1d_finite(values, "noise_for_psd")
    nperseg = int(round(nperseg_sec * sample_rate_hz))
    if nperseg <= 0 or nperseg > values.size:
        raise ValueError(f"nperseg={nperseg} is invalid for {values.size} samples.")
    noverlap = int(round(nperseg * overlap_fraction))
    if not 0 <= noverlap < nperseg:
        raise ValueError("Welch overlap must be between zero and nperseg-1.")
    frequencies_hz, raw_psd = signal.welch(
        values,
        fs=sample_rate_hz,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
        return_onesided=True,
        scaling="density",
    )
    if not np.all(np.diff(frequencies_hz) > 0):
        raise ValueError("PSD frequency bins are not strictly increasing.")
    if not np.isfinite(raw_psd).all() or np.any(raw_psd < 0):
        raise ValueError("PSD contains invalid values.")
    in_band = (frequencies_hz >= LOW_HZ) & (frequencies_hz <= HIGH_HZ)
    positive_reference = raw_psd[in_band & (raw_psd > 0)]
    if positive_reference.size == 0:
        raise ValueError("PSD has no positive values inside the analysis band.")
    floor = max(np.finfo(np.float64).tiny, floor_relative * float(np.median(positive_reference)))
    floored_psd = np.maximum(raw_psd, floor)
    return frequencies_hz, floored_psd, {
        "nperseg": nperseg,
        "noverlap": noverlap,
        "window": "hann-periodic",
        "detrend": "constant",
        "scaling": "density",
        "floor": floor,
        "floored_bins": int(np.sum(raw_psd < floor)),
    }

def save_npz_or_validate(path, **arrays):
    path = Path(path)
    if path.exists():
        with np.load(path, allow_pickle=False) as existing:
            if set(existing.files) != set(arrays):
                raise FileExistsError(f"Existing {path} has different keys and will not be overwritten.")
            for key, expected in arrays.items():
                if not np.array_equal(existing[key], np.asarray(expected)):
                    raise FileExistsError(f"Existing {path} differs at key {key!r} and will not be overwritten.")
        print("Validated existing artifact:", path)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)
    print("Saved new artifact:", path)

def save_json_or_validate(path, payload):
    path = Path(path)
    normalized = json.loads(json.dumps(payload, sort_keys=True, default=str))
    if path.exists():
        if json.loads(path.read_text()) != normalized:
            raise FileExistsError(f"Existing {path} has different content and will not be overwritten.")
        print("Validated existing metadata:", path)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(normalized, indent=2, sort_keys=True) + "\n")
    temporary.replace(path)
    print("Saved new metadata:", path)

rng = np.random.default_rng(SEED)
synthetic_duration_sec = 64
white_driver = rng.normal(scale=2e-21, size=synthetic_duration_sec * SAMPLE_RATE_HZ)
synthetic_reference_noise = signal.lfilter([1.0], [1.0, -0.96], white_driver)
synthetic_reference_noise += 1.5e-20 * np.sin(2 * np.pi * 60 * np.arange(synthetic_reference_noise.size) / SAMPLE_RATE_HZ)

synthetic_psd_frequencies_hz, synthetic_psd, synthetic_psd_settings = estimate_psd_welch(synthetic_reference_noise)
nperseg = synthetic_psd_settings["nperseg"]
noverlap = synthetic_psd_settings["noverlap"]
manual_frequencies_hz, manual_psd = manual_welch_psd(
    synthetic_reference_noise[: 16 * SAMPLE_RATE_HZ], SAMPLE_RATE_HZ, nperseg, noverlap
)
library_frequencies_hz, library_psd = signal.welch(
    synthetic_reference_noise[: 16 * SAMPLE_RATE_HZ], fs=SAMPLE_RATE_HZ,
    window="hann", nperseg=nperseg, noverlap=noverlap,
    detrend="constant", return_onesided=True, scaling="density",
)
assert np.array_equal(manual_frequencies_hz, library_frequencies_hz)
manual_library_relative_error = np.max(
    np.abs(manual_psd - library_psd) / np.maximum(library_psd, np.finfo(float).tiny)
)
assert manual_library_relative_error < 1e-10
integrated_power = np.trapz(library_psd, library_frequencies_hz)
variance_ratio = integrated_power / np.var(synthetic_reference_noise[: 16 * SAMPLE_RATE_HZ])
assert 0.85 < variance_ratio < 1.15

fig, ax = plt.subplots(figsize=(8.5, 4), constrained_layout=True)
ax.loglog(synthetic_psd_frequencies_hz[1:], np.sqrt(synthetic_psd[1:]))
ax.axvspan(LOW_HZ, HIGH_HZ, color="tab:blue", alpha=0.12, label="Analysis band")
ax.set(title="Synthetic noise amplitude spectral density", xlabel="Frequency (Hz)", ylabel="ASD (strain / sqrt(Hz))")
ax.legend()
plt.show()
print(f"Manual/SciPy Welch maximum relative error: {manual_library_relative_error:.3e}")
print(f"Integrated PSD / time-domain variance: {variance_ratio:.4f}")
print("PSD floor:", f"{synthetic_psd_settings['floor']:.3e}", "floored bins:", synthetic_psd_settings["floored_bins"])

def load_noise_handoffs(manifest_path):
    if not manifest_path.is_file():
        return {}, pd.DataFrame()
    manifest = pd.read_csv(manifest_path)
    required = {
        "noise_id", "group_id", "run", "sample_rate_hz", "n_samples",
        "strain_path", "split", "quality_status",
    }
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(f"Noise manifest is missing columns: {sorted(missing)}")
    if manifest["noise_id"].duplicated().any():
        raise ValueError("noise_id must be unique in the noise manifest.")
    accepted = manifest[manifest["quality_status"].astype(str).str.lower().isin({"accepted", "ok", "pass", "passed"})].copy()
    if accepted.empty:
        raise ValueError("Noise manifest contains no accepted rows.")
    sources = {}
    for row in accepted.itertuples(index=False):
        if float(row.sample_rate_hz) != SAMPLE_RATE_HZ:
            raise ValueError(f"{row.noise_id} has sample rate {row.sample_rate_hz}, expected {SAMPLE_RATE_HZ}.")
        strain_path = resolve_repo_path(row.strain_path)
        values = validate_1d_finite(np.load(strain_path, allow_pickle=False), f"noise[{row.noise_id}]")
        if values.size != int(row.n_samples):
            raise ValueError(f"{row.noise_id} manifest n_samples disagrees with {strain_path}.")
        sources[str(row.noise_id)] = {"strain": values, "metadata": row._asdict(), "path": strain_path}
    return sources, accepted

noise_sources, accepted_noise_manifest = load_noise_handoffs(NOISE_MANIFEST_PATH)
PSD_RECORDS = {
    "synthetic": {
        "frequency_hz": synthetic_psd_frequencies_hz,
        "psd_strain2_per_hz": synthetic_psd,
        "settings": synthetic_psd_settings,
    }
}
for noise_id, source in noise_sources.items():
    frequencies_hz, psd_values, settings = estimate_psd_welch(source["strain"])
    PSD_RECORDS[noise_id] = {
        "frequency_hz": frequencies_hz,
        "psd_strain2_per_hz": psd_values,
        "settings": settings,
    }
    psd_path = PREPROCESSING_DIR / "psd" / f"{noise_id}.npz"
    save_npz_or_validate(psd_path, frequency_hz=frequencies_hz, psd_strain2_per_hz=psd_values)
    save_json_or_validate(Path(str(psd_path) + ".json"), {
        "experiment_id": EXPERIMENT_ID,
        "noise_id": noise_id,
        "source_path": str(source["path"].relative_to(PROJECT_ROOT)),
        "sample_rate_hz": SAMPLE_RATE_HZ,
        "psd_units": "strain^2/Hz",
        **settings,
    })
print(f"Loaded {len(noise_sources)} real accepted noise source(s); cached {len(PSD_RECORDS) - 1} real PSD(s).")


**Cell 10**

### Your findings and contribution record

**Implementation status:** The notebook now contains an explicit Welch calculation and reconciles it against SciPy using the same Hann window, overlap, detrending and density normalization. It checks Parseval-scale power, strictly increasing frequency bins and finite nonnegative PSD values. Near-zero bins receive a recorded floor equal to a small fraction of the in-band median. When the Role 1 manifest exists, one PSD artifact and JSON settings sidecar are written per `noise_id` without overwriting conflicting files.

**Pending evidence:** Replace the synthetic ASD interpretation with measured O3a/O4a results after Role 1 supplies accepted source arrays and metadata.


**Cell 11**

## Task 3: Whiten without erasing physical signal

### Background knowledge
Whitening divides frequency components by the square root of a matching PSD, then transforms back. Constants depend on one-/two-sided PSD and FFT conventions. It is not feature-wise StandardScaler normalization, and an arbitrary unit-variance result alone does not prove correctness.

### Your task
Choose and explain a normalization convention. Interpolate the noise PSD to the buffer FFT grid and apply whitening yourself. Compare spectra before and after on a noise-only interval. Check finite values and reconstruction behavior. Preserve the PSD scale convention for all runs; explain any floor or zero-bin handling.

### Evidence and completion checks
A derivation in your words, before/after PSD and time plots, and empirical checks of frequency flattening.

**Cell 12 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 12
def whitening_scale(n_samples, sample_rate_hz, psd_frequencies_hz, psd_values):
    fft_frequencies_hz = np.fft.rfftfreq(n_samples, d=1 / sample_rate_hz)
    interpolated_psd = np.interp(fft_frequencies_hz, psd_frequencies_hz, psd_values)
    if not np.isfinite(interpolated_psd).all() or np.any(interpolated_psd <= 0):
        raise ValueError("Interpolated PSD must be finite and strictly positive.")
    # With NumPy's unnormalized forward FFT and one-sided PSD,
    # sqrt(2*dt/PSD) produces approximately unit-variance white noise.
    return fft_frequencies_hz, np.sqrt((2.0 / sample_rate_hz) / interpolated_psd)

def whiten_with_psd(values, sample_rate_hz, psd_frequencies_hz, psd_values):
    values = validate_1d_finite(values, "strain_to_whiten", minimum_length=8)
    _, scale = whitening_scale(values.size, sample_rate_hz, psd_frequencies_hz, psd_values)
    whitened = np.fft.irfft(np.fft.rfft(values) * scale, n=values.size)
    if not np.isfinite(whitened).all():
        raise ValueError("Whitened strain contains invalid values.")
    return whitened

synthetic_whitened = whiten_with_psd(
    synthetic_reference_noise, SAMPLE_RATE_HZ, synthetic_psd_frequencies_hz, synthetic_psd
)
_, inverse_scale = whitening_scale(
    synthetic_reference_noise.size, SAMPLE_RATE_HZ, synthetic_psd_frequencies_hz, synthetic_psd
)
reconstructed = np.fft.irfft(
    np.fft.rfft(synthetic_whitened) / inverse_scale, n=synthetic_reference_noise.size
)
reconstruction_relative_error = np.linalg.norm(reconstructed - synthetic_reference_noise) / np.linalg.norm(synthetic_reference_noise)
assert reconstruction_relative_error < 1e-10
assert np.isfinite(synthetic_whitened).all()

whitened_frequencies_hz, whitened_psd, _ = estimate_psd_welch(synthetic_whitened)
analysis_band = (synthetic_psd_frequencies_hz >= LOW_HZ) & (synthetic_psd_frequencies_hz <= HIGH_HZ)
before_db = 10 * np.log10(synthetic_psd[analysis_band])
after_db = 10 * np.log10(whitened_psd[analysis_band])
before_spread_db = np.percentile(before_db, 95) - np.percentile(before_db, 5)
after_spread_db = np.percentile(after_db, 95) - np.percentile(after_db, 5)
assert after_spread_db < before_spread_db

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
view = slice(0, int(1.0 * SAMPLE_RATE_HZ))
axes[0].plot(np.arange(view.stop) / SAMPLE_RATE_HZ, synthetic_reference_noise[view], label="Raw")
axes[0].set(title="Raw colored-noise interval", xlabel="Time (s)", ylabel="Strain")
axes[1].semilogy(synthetic_psd_frequencies_hz[analysis_band], synthetic_psd[analysis_band], label="Before whitening")
axes[1].semilogy(whitened_frequencies_hz[analysis_band], whitened_psd[analysis_band], label="After whitening")
axes[1].set(title="PSD flattening inside the analysis band", xlabel="Frequency (Hz)", ylabel="PSD (output units^2 / Hz)")
axes[1].legend()
plt.show()

print(f"Whitened standard deviation: {synthetic_whitened.std():.4f}")
print(f"In-band 5th-95th percentile PSD spread: {before_spread_db:.2f} dB before, {after_spread_db:.2f} dB after")
print(f"Whiten/inverse reconstruction relative error: {reconstruction_relative_error:.3e}")


**Cell 13**

### Your findings and contribution record

**Implementation status:** Whitening uses the stated one-sided-PSD convention `rFFT(x) * sqrt(2*dt/PSD)` and validates finite output, approximate in-band flattening and numerical inversion. The cell reports standard deviation, PSD-spread reduction and reconstruction error instead of treating unit variance alone as proof of correctness.

**Pending evidence:** Repeat the same diagnostics for at least one accepted source segment from each observing run; retain the run labels and `noise_id` values in the interpretation.


**Cell 14**

## Task 4: Apply and validate the bandpass

### Background knowledge
A bandpass removes frequencies outside the analysis band. Forward/backward filtering reduces phase delay but has boundary effects; the effective response is applied twice. Second-order sections can improve numerical stability. Zero-phase filtering is an offline operation.

### Your task
Design a fourth-order Butterworth 35-350 Hz bandpass for fs=4096 Hz. Plot its response. Test a below-band, in-band and above-band sinusoid and measure attenuation. Inspect both edges and decide how much padding is needed. Record design order and forward/backward behavior.

### Evidence and completion checks
Response plot, three-sinusoid check, edge inspection and exact settings.

**Cell 15 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 15
def design_bandpass_sos(sample_rate_hz=SAMPLE_RATE_HZ, low_hz=LOW_HZ,
                        high_hz=HIGH_HZ, order=FILTER_ORDER):
    if not 0 < low_hz < high_hz < sample_rate_hz / 2:
        raise ValueError("Bandpass edges must satisfy 0 < low < high < Nyquist.")
    return signal.butter(order, [low_hz, high_hz], btype="bandpass", fs=sample_rate_hz, output="sos")

def bandpass_zero_phase(values, sample_rate_hz=SAMPLE_RATE_HZ):
    values = validate_1d_finite(values, "strain_to_bandpass", minimum_length=64)
    filtered = signal.sosfiltfilt(design_bandpass_sos(sample_rate_hz), values)
    if not np.isfinite(filtered).all():
        raise ValueError("Bandpassed strain contains invalid values.")
    return filtered

def measured_tone_amplitude(values, frequency_hz, sample_rate_hz):
    values = np.asarray(values)
    times = np.arange(values.size) / sample_rate_hz
    cosine = np.cos(2 * np.pi * frequency_hz * times)
    sine = np.sin(2 * np.pi * frequency_hz * times)
    return 2 * np.hypot(values @ cosine, values @ sine) / values.size

sos = design_bandpass_sos()
response_frequencies_hz, response = signal.sosfreqz(sos, worN=16384, fs=SAMPLE_RATE_HZ)
effective_gain = np.abs(response) ** 2  # sosfiltfilt applies the response forward and backward.

validation_duration_sec = 8
validation_time_sec = np.arange(validation_duration_sec * SAMPLE_RATE_HZ) / SAMPLE_RATE_HZ
validation_frequencies_hz = [10.0, 100.0, 600.0]
composite = sum(np.sin(2 * np.pi * frequency * validation_time_sec) for frequency in validation_frequencies_hz)
filtered_composite = bandpass_zero_phase(composite)
center = slice(2 * SAMPLE_RATE_HZ, 6 * SAMPLE_RATE_HZ)
tone_gains = {}
for frequency_hz in validation_frequencies_hz:
    input_amplitude = measured_tone_amplitude(composite[center], frequency_hz, SAMPLE_RATE_HZ)
    output_amplitude = measured_tone_amplitude(filtered_composite[center], frequency_hz, SAMPLE_RATE_HZ)
    tone_gains[frequency_hz] = output_amplitude / input_amplitude

assert tone_gains[10.0] < 0.05
assert tone_gains[100.0] > 0.90
assert tone_gains[600.0] < 0.05

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].semilogx(response_frequencies_hz[1:], 20 * np.log10(np.maximum(effective_gain[1:], 1e-12)))
axes[0].axvline(LOW_HZ, color="black", linestyle="--", linewidth=1)
axes[0].axvline(HIGH_HZ, color="black", linestyle="--", linewidth=1)
axes[0].set(title="Effective zero-phase Butterworth response", xlabel="Frequency (Hz)", ylabel="Gain (dB)", ylim=(-100, 5))
labels = [f"{frequency:g} Hz" for frequency in validation_frequencies_hz]
axes[1].bar(labels, [tone_gains[frequency] for frequency in validation_frequencies_hz], color="tab:blue")
axes[1].axhline(1, color="black", linewidth=1)
axes[1].set(title="Measured sinusoid transmission", xlabel="Test frequency", ylabel="Output / input amplitude")
plt.show()

print("Measured gains:", {f"{key:g} Hz": round(value, 6) for key, value in tone_gains.items()})
print(f"Edge policy: process at least {REQUIRED_PADDING_SEC:.0f} s on each side before extracting the central {FINAL_WINDOW_SEC:.0f} s crop.")


**Cell 16**

### Your findings and contribution record

**Implementation status:** The bandpass is implemented as fourth-order Butterworth second-order sections and applied with `sosfiltfilt`. The cell plots the effective forward/backward response and measures transmission for below-band, in-band and above-band sinusoids. It records a fifteen-second context policy so filtering occurs before the central crop.

**Pending evidence:** Inspect the boundary region on a real padded injection and record whether the agreed padding is conservative enough for every waveform duration used by Role 2.


**Cell 17**

## Task 5: Crop only after processing padded buffers

### Background knowledge
The project convention is two seconds before plus two seconds after merger: four seconds total, or 16384 samples at 4096 Hz. Cropping first can create artificial filter/FFT boundaries near the signal. Local buffer indices differ from GPS/source indices.

### Your task
Use Role 2 local merger indices to crop after whitening and filtering. Verify the peak alignment on one known injection. Check exact length, finiteness and enough padding for your documented edge policy. Reject invalid bounds rather than silently truncate. Explain what happens for a near-edge candidate.

### Evidence and completion checks
One final input plot with merger marker, sample-count proof and a rejection ledger.

**Cell 18 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 18
def preprocess_padded_buffer(values, merger_index, psd_frequencies_hz, psd_values,
                             sample_rate_hz=SAMPLE_RATE_HZ):
    values = validate_1d_finite(values, "padded_injection_buffer", minimum_length=WINDOW_SAMPLES)
    merger_index = int(merger_index)
    required_context = int(REQUIRED_PADDING_SEC * sample_rate_hz)
    if merger_index < required_context or values.size - merger_index < required_context:
        raise ValueError(
            f"Merger index {merger_index} lacks {REQUIRED_PADDING_SEC:g} s of context on both sides."
        )
    whitened = whiten_with_psd(values, sample_rate_hz, psd_frequencies_hz, psd_values)
    filtered = bandpass_zero_phase(whitened, sample_rate_hz)
    half_window = int(HALF_WINDOW_SEC * sample_rate_hz)
    crop_start = merger_index - half_window
    crop_stop = merger_index + half_window
    if crop_start < 0 or crop_stop > values.size:
        raise ValueError("Requested crop exceeds the supplied buffer; refusing to truncate.")
    cropped = filtered[crop_start:crop_stop].copy()
    if cropped.shape != (WINDOW_SAMPLES,) or not np.isfinite(cropped).all():
        raise ValueError(f"Invalid processed crop shape or values: {cropped.shape}.")
    return {
        "raw": values,
        "whitened": whitened,
        "filtered": filtered,
        "cropped": cropped,
        "crop_start": crop_start,
        "crop_stop": crop_stop,
    }

# Synthetic alignment and edge-rejection check.
synthetic_buffer_samples = int(2 * REQUIRED_PADDING_SEC * SAMPLE_RATE_HZ)
synthetic_raw_buffer = synthetic_reference_noise[:synthetic_buffer_samples].copy()
synthetic_merger_index = synthetic_buffer_samples // 2
waveform_time_sec = np.arange(int(2.5 * SAMPLE_RATE_HZ)) / SAMPLE_RATE_HZ
synthetic_waveform = 5e-20 * signal.chirp(
    waveform_time_sec, f0=45, f1=300, t1=waveform_time_sec[-1], method="quadratic"
)
synthetic_waveform *= np.linspace(0.05, 1.0, synthetic_waveform.size)
waveform_peak_index = int(np.argmax(np.abs(synthetic_waveform)))
waveform_start = synthetic_merger_index - waveform_peak_index
synthetic_raw_buffer[waveform_start:waveform_start + synthetic_waveform.size] += synthetic_waveform
assert waveform_start + waveform_peak_index == synthetic_merger_index

synthetic_result = preprocess_padded_buffer(
    synthetic_raw_buffer,
    synthetic_merger_index,
    synthetic_psd_frequencies_hz,
    synthetic_psd,
)
assert synthetic_result["cropped"].shape == (WINDOW_SAMPLES,)

rejection_rows = []
try:
    preprocess_padded_buffer(
        synthetic_raw_buffer,
        int(REQUIRED_PADDING_SEC * SAMPLE_RATE_HZ) - 1,
        synthetic_psd_frequencies_hz,
        synthetic_psd,
    )
    raise AssertionError("Near-edge buffer should have been rejected.")
except ValueError as exc:
    rejection_rows.append({"sample_id": "synthetic-near-edge", "rejection_reason": str(exc)})

crop_time_sec = np.arange(WINDOW_SAMPLES) / SAMPLE_RATE_HZ - HALF_WINDOW_SEC
context_start = synthetic_merger_index - int(3 * SAMPLE_RATE_HZ)
context_stop = synthetic_merger_index + int(3 * SAMPLE_RATE_HZ)
context_time_sec = np.arange(context_start, context_stop) / SAMPLE_RATE_HZ - synthetic_merger_index / SAMPLE_RATE_HZ
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
axes[0, 0].plot(context_time_sec, synthetic_result["raw"][context_start:context_stop])
axes[0, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set(title="Injected padded buffer", xlabel="Time from merger (s)", ylabel="Strain")
axes[0, 1].plot(context_time_sec, synthetic_result["whitened"][context_start:context_stop])
axes[0, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0, 1].set(title="After PSD whitening", xlabel="Time from merger (s)", ylabel="Whitened amplitude")
axes[1, 0].plot(context_time_sec, synthetic_result["filtered"][context_start:context_stop])
axes[1, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set(title="After 35-350 Hz zero-phase bandpass", xlabel="Time from merger (s)", ylabel="Filtered amplitude")
axes[1, 1].plot(crop_time_sec, synthetic_result["cropped"])
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1, label="Aligned waveform peak")
axes[1, 1].set(title="Final merger-centered model input", xlabel="Time from merger (s)", ylabel="Processed amplitude")
axes[1, 1].legend()
plt.show()

print("Synthetic merger index:", synthetic_merger_index)
print("Synthetic final crop:", synthetic_result["cropped"].shape)
print("Synthetic edge rejection:", rejection_rows[0]["rejection_reason"])

def load_injection_manifest(path):
    if not path.is_file():
        return pd.DataFrame()
    manifest = pd.read_csv(path)
    required = {
        "sample_id", "noise_id", "group_id", "run", "split", "buffer_path",
        "buffer_start_in_noise_samples", "merger_index_in_buffer", "status",
    }
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(f"Injection manifest is missing columns: {sorted(missing)}")
    if manifest["sample_id"].duplicated().any():
        raise ValueError("sample_id must be unique in the injection manifest.")
    accepted = manifest[manifest["status"].astype(str).str.lower().isin({"accepted", "ok", "pass", "passed"})].copy()
    if accepted.empty:
        raise ValueError("Injection manifest contains no accepted rows.")
    group_split_counts = accepted.groupby("group_id")["split"].nunique()
    if (group_split_counts > 1).any():
        leaking = group_split_counts[group_split_counts > 1].index.tolist()
        raise ValueError(f"Source groups cross splits: {leaking}")
    if ((accepted["run"] == "O4a") & (accepted["split"] != "O4a_test")).any():
        raise ValueError("Every O4a record must remain in O4a_test.")
    return accepted

accepted_injections = load_injection_manifest(INJECTION_MANIFEST_PATH)
processed_rows = []
if not accepted_injections.empty:
    for row in accepted_injections.itertuples(index=False):
        sample_id = str(row.sample_id)
        noise_id = str(row.noise_id)
        try:
            if noise_id not in PSD_RECORDS:
                raise KeyError(f"No PSD artifact is available for noise_id={noise_id}.")
            buffer_path = resolve_repo_path(row.buffer_path)
            buffer_values = np.load(buffer_path, allow_pickle=False)
            psd_record = PSD_RECORDS[noise_id]
            result = preprocess_padded_buffer(
                buffer_values,
                int(row.merger_index_in_buffer),
                psd_record["frequency_hz"],
                psd_record["psd_strain2_per_hz"],
            )
            absolute_crop_start = (
                int(row.buffer_start_in_noise_samples) + result["crop_start"]
            )
            median = float(np.median(result["cropped"]))
            mad = float(np.median(np.abs(result["cropped"] - median)))
            processed_rows.append({
                "sample_id": sample_id,
                "noise_id": noise_id,
                "group_id": str(row.group_id),
                "run": str(row.run),
                "split": str(row.split),
                "source_buffer_path": str(buffer_path.relative_to(PROJECT_ROOT)),
                "crop_start_in_noise_samples": absolute_crop_start,
                "crop_stop_in_noise_samples": absolute_crop_start + WINDOW_SAMPLES,
                "X": result["cropped"],
                "mean": float(np.mean(result["cropped"])),
                "std": float(np.std(result["cropped"])),
                "rms": float(np.sqrt(np.mean(result["cropped"] ** 2))),
                "peak_abs": float(np.max(np.abs(result["cropped"]))),
                "mad": mad,
                "peak_to_mad": float(np.max(np.abs(result["cropped"] - median)) / max(mad, np.finfo(float).tiny)),
            })
        except Exception as exc:
            rejection_rows.append({"sample_id": sample_id, "rejection_reason": f"{type(exc).__name__}: {exc}"})
print(f"Real accepted injections discovered: {len(accepted_injections)}")
print(f"Real processed rows: {len(processed_rows)}; rejected during preprocessing: {len(rejection_rows) - 1}")


**Cell 19**

### Your findings and contribution record

**Implementation status:** `preprocess_padded_buffer` validates context, whitens and filters the full buffer, and only then extracts exactly 16,384 samples centered on the merger index. A synthetic waveform peak is aligned explicitly, and a near-edge candidate is rejected rather than truncated. The real-data path preserves source indices and records numerical quality summaries and rejection reasons by `sample_id`.

**Pending evidence:** The real path remains conditional on Role 2's injection manifest, buffers and Role 1/3 PSD records.


**Cell 20**

## Task 6: Assemble data without changing labels or splits

### Background knowledge
A target row must stay paired with the same observation through all filtering and reordering. Training-only ML scaling belongs to the model workflow; per-segment background PSD estimation is signal processing and must use a consistent inference-time policy.

### Your task
Join buffers and labels by sample_id. Preserve group and split. Export X, theta and ID/split arrays plus a row manifest. Audit every join and give actual per-split counts. Do not add feature scaling fitted across all rows or normalize each waveform amplitude casually; amplitude helps infer distance.

### Evidence and completion checks
Role 4 can load fixed-shape inputs and labels and verify ID order without using your functions.

**Cell 21 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 21
def save_text_or_validate(path, text):
    path = Path(path)
    if path.exists():
        if path.read_text() != text:
            raise FileExistsError(f"Existing {path} differs and will not be overwritten.")
        print("Validated existing artifact:", path)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text)
    temporary.replace(path)
    print("Saved new artifact:", path)

processed_manifest = pd.DataFrame()
split_counts = pd.DataFrame()
if processed_rows:
    targets = pd.read_csv(TARGETS_PATH)
    required_target_columns = {"sample_id", *TARGET_NAMES}
    missing_targets = required_target_columns - set(targets.columns)
    if missing_targets:
        raise ValueError(f"Targets table is missing columns: {sorted(missing_targets)}")
    if targets["sample_id"].duplicated().any():
        raise ValueError("targets.csv contains duplicate sample_id values.")

    metadata_columns = [key for key in processed_rows[0] if key != "X"]
    processed_manifest = pd.DataFrame([{key: row[key] for key in metadata_columns} for row in processed_rows])
    processed_manifest.insert(0, "row_index", np.arange(len(processed_manifest), dtype=np.int64))
    processed_manifest["n_samples"] = WINDOW_SAMPLES
    processed_manifest["finite_status"] = True
    if processed_manifest["sample_id"].duplicated().any():
        raise ValueError("Processed rows contain duplicate sample_id values.")

    joined = processed_manifest.merge(
        targets[["sample_id", *TARGET_NAMES]], on="sample_id", how="left", validate="one_to_one", indicator=True
    )
    if not (joined["_merge"] == "both").all():
        missing_ids = joined.loc[joined["_merge"] != "both", "sample_id"].tolist()
        raise ValueError(f"Processed rows lack target labels: {missing_ids}")
    joined = joined.drop(columns="_merge")
    if joined[TARGET_NAMES].isna().any().any():
        raise ValueError("Target table contains missing values for retained processed rows.")

    # Final crop windows may not overlap within a source noise record.
    for noise_id, group in joined.groupby("noise_id"):
        intervals = group[["crop_start_in_noise_samples", "crop_stop_in_noise_samples"]].sort_values(
            "crop_start_in_noise_samples"
        ).to_numpy()
        if len(intervals) > 1 and np.any(intervals[1:, 0] < intervals[:-1, 1]):
            raise ValueError(f"Final crop windows overlap within noise_id={noise_id}.")

    X = np.stack([row["X"] for row in processed_rows]).astype(np.float64, copy=False)
    theta = joined[TARGET_NAMES].to_numpy(dtype=np.float64)
    if X.shape != (len(joined), WINDOW_SAMPLES) or theta.shape != (len(joined), len(TARGET_NAMES)):
        raise ValueError(f"Unexpected final shapes: X={X.shape}, theta={theta.shape}")
    if not np.isfinite(X).all() or not np.isfinite(theta).all():
        raise ValueError("Final arrays contain invalid values.")

    processed_path = PREPROCESSING_DIR / "processed.npz"
    save_npz_or_validate(
        processed_path,
        X=X,
        theta=theta,
        sample_id=joined["sample_id"].to_numpy(dtype=str),
        split=joined["split"].to_numpy(dtype=str),
        noise_id=joined["noise_id"].to_numpy(dtype=str),
        group_id=joined["group_id"].to_numpy(dtype=str),
        run=joined["run"].to_numpy(dtype=str),
    )
    save_text_or_validate(
        PREPROCESSING_DIR / "processed_manifest.csv",
        joined.to_csv(index=False),
    )
    real_rejections = pd.DataFrame(rejection_rows[1:], columns=["sample_id", "rejection_reason"])
    save_text_or_validate(
        PREPROCESSING_DIR / "rejections.csv",
        real_rejections.to_csv(index=False),
    )
    processing_settings = {
        "experiment_id": EXPERIMENT_ID,
        "sample_rate_hz": SAMPLE_RATE_HZ,
        "final_window_sec": FINAL_WINDOW_SEC,
        "window_samples": WINDOW_SAMPLES,
        "required_padding_sec_each_side": REQUIRED_PADDING_SEC,
        "psd": {
            "method": "Welch",
            "nperseg_sec": PSD_NPERSEG_SEC,
            "overlap_fraction": PSD_OVERLAP_FRACTION,
            "window": "hann-periodic",
            "detrend": "constant",
            "scaling": "density",
            "floor_relative_to_in_band_median": PSD_FLOOR_RELATIVE,
        },
        "whitening": "rfft multiplied by sqrt(2*dt/interpolated_one_sided_psd)",
        "bandpass": {
            "low_hz": LOW_HZ,
            "high_hz": HIGH_HZ,
            "prototype_order": FILTER_ORDER,
            "representation": "second-order sections",
            "application": "zero-phase sosfiltfilt",
        },
        "target_order": TARGET_NAMES,
        "feature_scaling": "none in preprocessing; retain amplitude information",
        "categorical_encoding": "none; run/split/group remain metadata only",
        "software": {
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "numpy": np.__version__,
            "scipy": __import__("scipy").__version__,
            "pandas": pd.__version__,
        },
    }
    save_json_or_validate(PREPROCESSING_DIR / "processing_settings.json", processing_settings)

    split_counts = joined.groupby(["run", "split"], dropna=False).agg(
        final_examples=("sample_id", "size"),
        source_groups=("group_id", "nunique"),
        noise_sources=("noise_id", "nunique"),
    ).reset_index()
    print(split_counts.to_string(index=False))
    print("Final arrays:", X.shape, theta.shape)
else:
    print("No real Role 2 buffers were processed, so no pilot dataset was exported.")
    print("Expected inputs:", INJECTION_MANIFEST_PATH, "and", TARGETS_PATH)


**Cell 22**

### Your findings and contribution record

**Implementation status:** The assembly cell joins labels by unique `sample_id`, requires one-to-one coverage, preserves `noise_id`, `group_id`, run and split, verifies group-disjoint splits and nonoverlapping final crops, and writes explicit-dtype NPZ/CSV/JSON artifacts. Existing identical artifacts are validated and reused; conflicting artifacts are never overwritten. No training-wide or per-example amplitude scaling is performed.

**Pending evidence:** Actual split counts are printed only after real manifests exist; unknown counts are never filled with zero.


**Cell 23**

## Task 7: Explain the evidence for the milestone

### Background knowledge
A processing description must distinguish invalid samples, valid loud observations, normalization, feature selection and categorical encoding. It should explain how the transformations affect the scientific signal.

### Your task
Build one multipanel raw/whitened/filtered/cropped figure, with units and matching time axes where possible. Write 150 words covering missing values, outliers, whitening, filtering, cropping and numerical issues. State no categorical encoding if inputs are numeric, and no learned feature selection if none was performed. Keep your contribution log.

### Evidence and completion checks
A self-contained preprocessing section, labeled figure, settings file and explicit completed/planned status.

**Cell 24 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [ ]:
# Cell 24
evidence_crop = synthetic_result["cropped"]
evidence_frequencies_hz, evidence_psd, _ = estimate_psd_welch(evidence_crop)
evidence_band = (evidence_frequencies_hz >= LOW_HZ) & (evidence_frequencies_hz <= HIGH_HZ)
evidence_time_sec = np.arange(WINDOW_SAMPLES) / SAMPLE_RATE_HZ - HALF_WINDOW_SEC

fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
raw_context = synthetic_result["raw"][context_start:context_stop]
axes[0, 0].plot(context_time_sec, raw_context)
axes[0, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set(title="Raw injected buffer near merger", xlabel="Time from merger (s)", ylabel="Strain")
axes[0, 1].plot(context_time_sec, synthetic_result["whitened"][context_start:context_stop])
axes[0, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0, 1].set(title="Whitened padded buffer", xlabel="Time from merger (s)", ylabel="Whitened amplitude")
axes[1, 0].plot(evidence_time_sec, evidence_crop)
axes[1, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set(title="Final 4-second processed input", xlabel="Time from merger (s)", ylabel="Processed amplitude")
axes[1, 1].semilogy(evidence_frequencies_hz[evidence_band], evidence_psd[evidence_band])
axes[1, 1].set(title="Final input PSD inside analysis band", xlabel="Frequency (Hz)", ylabel="PSD (output units^2 / Hz)")
plt.show()

synthetic_qc = pd.DataFrame([{
    "example": "synthetic validation",
    "n_samples": evidence_crop.size,
    "finite_fraction": float(np.isfinite(evidence_crop).mean()),
    "mean": float(evidence_crop.mean()),
    "std": float(evidence_crop.std()),
    "rms": float(np.sqrt(np.mean(evidence_crop**2))),
    "peak_abs": float(np.max(np.abs(evidence_crop))),
}])
print(synthetic_qc.to_string(index=False))
if not processed_manifest.empty:
    print("\nReal pilot QC by split:")
    print(processed_manifest.groupby(["run", "split"])[["std", "rms", "peak_abs", "peak_to_mad"]].agg(
        ["count", "median", "max"]
    ).round(4).to_string())

status_sentence = (
    f"The real pilot retained {len(processed_manifest)} examples across {processed_manifest['group_id'].nunique()} source groups."
    if not processed_manifest.empty
    else "The real Role 1/2 handoffs are not present, so only the synthetic validation has been completed."
)
preprocessing_summary = f"""
Preprocessing rejects empty, wrong-shaped, nonfinite, under-padded, or out-of-bounds buffers rather than imputing them. A four-second Hann-windowed Welch estimate from source noise produces a one-sided PSD in strain squared per hertz; a documented positive floor prevents division by zero. Whitening uses a consistent one-sided-PSD FFT normalization, after which a fourth-order Butterworth bandpass is applied as numerically stable second-order sections in forward and reverse directions. Every buffer retains fifteen seconds of context on each side before a merger-centered four-second crop is extracted. No per-example amplitude normalization is applied because amplitude carries distance information. No categorical encoding or learned feature selection occurs: run, split, noise, and group identifiers remain metadata. Loud examples are retained when finite and valid; peak-to-MAD and related summaries flag them for review instead of clipping them silently. {status_sentence}
""".strip()
print("\nMilestone preprocessing draft:\n")
print(preprocessing_summary)

completion_checklist = pd.DataFrame([
    ["Known-frequency FFT check", "complete", f"peak error {peak_error_hz:.3f} Hz"],
    ["Manual versus SciPy Welch", "complete", f"max relative error {manual_library_relative_error:.2e}"],
    ["Whitening flattening/reconstruction", "complete", f"reconstruction error {reconstruction_relative_error:.2e}"],
    ["Bandpass transmission", "complete", str({k: round(v, 4) for k, v in tone_gains.items()})],
    ["Exact crop and edge rejection", "complete", f"shape {synthetic_result['cropped'].shape}"],
    ["Real source PSD export", "complete" if noise_sources else "pending", f"{len(noise_sources)} real noise source(s)"],
    ["Real processed dataset handoff", "complete" if not processed_manifest.empty else "pending", f"{len(processed_manifest)} retained row(s)"],
], columns=["check", "status", "evidence"])
print("\nCompletion checklist:\n")
print(completion_checklist.to_string(index=False))


**Cell 25**

### Your findings and contribution record

**Implementation status:** The final cell produces a self-contained raw/whitened/filtered/cropped evidence figure, bounded QC summaries, a milestone-ready preprocessing paragraph and an explicit completion checklist. Synthetic validations are labeled as such. Real retained/rejected counts and observing-run interpretations remain pending until the shared pilot artifacts are available.

**Contribution record:** Record the eventual notebook owner and execution date here after the team assigns ownership. Do not replace “pending” items with claims until the real outputs have been generated and reviewed with Roles 1, 2 and 4.


**Cell 26**

## Milestone completion and handoff

Before marking your role complete, show the teammate consuming your artifacts one real record and its provenance. Give them a settings file, retained/rejected counts and a short interpretation, not just an array. Check sample_id, noise_id, group_id, run, split and shape at the handoff. Save outputs under your role-owned folder for the same experiment_id.

Deliver your completed notebook, a contribution log, 100-150 words of findings, figures with descriptive titles/axes/units/legends/captions and an explicit list of unfinished checks. Do not copy another person's implementation into your notebook. Preserve the older notebooks and existing PDFs.

**Stop at this checkpoint:** dataset preparation, validated preprocessing and EDA, exact allocation tables and the three-algorithm methodology plan. Full linear/Ridge/SNPE training, final model ranking, posterior calibration results and 800-per-run manual regeneration are not mandatory tasks in this set. Feature importance remains an acknowledged rubric gap if deferred; the team must decide how to complete that requirement before submission rather than asserting full compliance.

**Write a final reflection:** What did you personally implement? What evidence supports it? Which decisions need another owner's agreement? Which findings belong to the new pilot rather than the historical project? What remains before submission?


**Cell 27**

## Background references

- [GWOSC data](https://gwosc.org/data/)
- [PyCBC waveforms](https://pycbc.org/pycbc/latest/html/waveform.html)
- [SciPy signal processing](https://docs.scipy.org/doc/scipy/reference/signal.html)
- [sbi calibration guide](https://sbi.readthedocs.io/en/latest/how_to_guide/16_sbc.html)

Consult library documentation for concepts and conventions. Your notebook should contain your own implementation and interpretation.